# Notebook 02: Extract Protein-Coding Genes

Load the Hubstenberger et al. mmc3.xlsx dataset, map Ensembl IDs to gene biotype
using `mygene` (Python equivalent of R's `org.Hs.eg.db`), and filter to protein-coding genes.

**Key decisions:**
- Use `mygene` API for biotype — queries NCBI/Ensembl like Jason's `org.Hs.eg.db`
- GENCODE v19 GTF is still used for gene lengths (NB03) and CAGE annotation (NB05), just not for biotype
- Strip Ensembl version suffixes while preserving PAR annotations

**Expected output:** ~15,465 protein-coding genes (superset of the 12,544 in Jason's final CSV)

In [1]:
import pandas as pd
import numpy as np
import mygene
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
REFERENCE_DIR = Path("../reference")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Hubstenberger mmc3.xlsx

In [2]:
# Load the Hubstenberger supplementary table
hub = pd.read_excel(RAW_DIR / "mmc3.xlsx")

print(f"Rows: {len(hub):,}")
print(f"Columns: {len(hub.columns)}")
print(f"\nColumn names:")
for i, col in enumerate(hub.columns):
    print(f"  {i}: {col}")

Rows: 28,321
Columns: 19

Column names:
  0: Ensembl Gene ID
  1: Associated Gene Name
  2: RNA enrichment in P-body (log2) (Fold change=sorted P-bodies/pre-sorted fraction)
  3: p-value of RNA enrichment 
  4: adjusted p-value (FDR) of RNA enrichment 
  5: sorted P-body replicate 1 (count per million reads (CPM))
  6: sorted P-body replicate 2 (count per million reads (CPM))
  7: sorted P-body replicate 3 (count per million reads (CPM))
  8: sorted P-body replicate average (count per million reads (CPM))
  9: pre-sorted fraction replicate 1 (count per million reads (CPM))
  10: pre-sorted fraction replicate 2 (count per million reads (CPM))
  11: pre-sorted fraction replicate 3 (count per million reads (CPM))
  12: pre-sorted fraction replicate average (count per million reads (CPM))
  13: sorted P-body replicate 1 (mapped read number)
  14: sorted P-body replicate 2 (mapped read number)
  15: sorted P-body replicate 3 (mapped read number)
  16: pre-sorted fraction replicate 1 (mapped

In [3]:
# Inspect first few rows and identify the Ensembl ID column
hub.head()

,Ensembl Gene ID,Associated Gene Name,RNA enrichment in P-body (log2) (Fold change=sorted P-bodies/pre-sorted fraction),p-value of RNA enrichment,adjusted p-value (FDR) of RNA enrichment,sorted P-body replicate 1 (count per million reads (CPM)),sorted P-body replicate 2 (count per million reads (CPM)),sorted P-body replicate 3 (count per million reads (CPM)),sorted P-body replicate average (count per million reads (CPM)),pre-sorted fraction replicate 1 (count per million reads (CPM)),pre-sorted fraction replicate 2 (count per million reads (CPM)),pre-sorted fraction replicate 3 (count per million reads (CPM)),pre-sorted fraction replicate average (count per million reads (CPM)),sorted P-body replicate 1 (mapped read number),sorted P-body replicate 2 (mapped read number),sorted P-body replicate 3 (mapped read number),pre-sorted fraction replicate 1 (mapped read number),pre-sorted fraction replicate 2 (mapped read number),pre-sorted fraction replicate 3 (mapped read number)
0,ENSG00000226532,RP5-1052M9.1,-8.831209,3.076139e-13,1.808141e-12,0.0,0.0,0.0,0.001,1.493488,1.24259504231823,0.887547,1.207877,0,0,0,56.0,60.0,49.0
1,ENSG00000248870,CTD-2015A6.2,-8.379843,2.546222e-06,9.473069e-06,0.0,0.0,0.0,0.001,0.000000,2.40235041514857,0.000000,0.800783,0,0,0,0.0,116.0,0.0
2,ENSG00000242173,ARHGDIG,-8.256319,3.590155e-10,1.768537e-09,0.0,0.0,0.0,0.001,1.360141,0.227809091091675,1.177358,0.921769,0,0,0,51.0,11.0,65.0
3,ENSG00000244411,KRTAP5-7,-8.201971,1.240737e-05,4.335307e-05,0.0,0.0,0.0,0.001,0.000000,2.1745413240569,0.000000,0.724847,0,0,0,0.0,105.0,0.0
4,ENSG00000262074,SNORD3B-2,-8.191104,1.120709e-09,5.330616e-09,0.0,0.0,0.0,0.001,0.853422,0.89052644699473,0.543396,0.762448,0,0,0,32.0,43.0,30.0


## 2. Strip Ensembl Version Suffixes

GENCODE IDs include version suffixes (e.g., `ENSG00000123456.7`). We strip these
to match the format used in Jason's reference data. PAR (pseudoautosomal region)
gene suffixes like `_PAR_Y` are preserved to avoid ID collisions.

In [4]:
def strip_ensembl_version(eid: str) -> str:
    """Strip version suffix from Ensembl ID, preserving PAR annotation."""
    if not isinstance(eid, str):
        return eid
    base, *rest = eid.split(".")
    par_suffix = ""
    if rest:
        version_part = rest[0]
        par_idx = version_part.find("_PAR_")
        if par_idx != -1:
            par_suffix = version_part[par_idx:]
    return base + par_suffix


# Identify the Ensembl ID column (may be named differently)
ensembl_col = None
for col in hub.columns:
    if 'ensembl' in col.lower() and 'id' in col.lower():
        ensembl_col = col
        break

if ensembl_col is None:
    # Try the first column as fallback
    ensembl_col = hub.columns[0]
    print(f"WARNING: No obvious Ensembl ID column found. Using first column: '{ensembl_col}'")
else:
    print(f"Ensembl ID column: '{ensembl_col}'")

# Strip version suffixes
hub["ensembl_id"] = hub[ensembl_col].apply(strip_ensembl_version)

# Validate uniqueness
n_unique = hub["ensembl_id"].nunique()
n_total = len(hub)
print(f"Total rows: {n_total:,}")
print(f"Unique Ensembl IDs after stripping: {n_unique:,}")
if n_unique < n_total:
    dupes = hub[hub["ensembl_id"].duplicated(keep=False)]
    print(f"WARNING: {n_total - n_unique} duplicate IDs found after version stripping!")
    print(dupes[[ensembl_col, "ensembl_id"]].head(10))

Ensembl ID column: 'Ensembl Gene ID'
Total rows: 28,321
Unique Ensembl IDs after stripping: 28,320
Empty DataFrame
Columns: [Ensembl Gene ID, ensembl_id]
Index: []


## 3. Map Gene Biotype via mygene (NCBI/Ensembl)

Query the MyGene.info API for `type_of_gene` — this is the Python equivalent of
R's `org.Hs.eg.db` which Jason used. Both draw from NCBI Gene, so results should
be nearly identical.

In [5]:
mg = mygene.MyGeneInfo()

all_ids = hub["ensembl_id"].dropna().unique().tolist()
print(f"Querying mygene for {len(all_ids):,} Ensembl IDs...")

# Batch query — scopes="ensembl.gene" maps Ensembl IDs to NCBI gene records
results = mg.querymany(
    all_ids,
    scopes="ensembl.gene",
    fields="type_of_gene,symbol",
    species="human",
    returnall=True
)

# Build biotype mapping
biotype_map = {}
for r in results["out"]:
    qid = r["query"]
    if r.get("notfound", False):
        continue
    gene_type = r.get("type_of_gene", None)
    if gene_type:
        biotype_map[qid] = gene_type

# Summary
from collections import Counter
mapped = len(biotype_map)
not_found = len([r for r in results["out"] if r.get("notfound", False)])
dupes = len(results.get("dup", []))
print(f"Mapped: {mapped:,} / {len(all_ids):,}")
print(f"Not found in mygene: {not_found:,}")
print(f"Duplicate hits: {dupes}")

print(f"\nBiotype distribution:")
for bt, count in Counter(biotype_map.values()).most_common(15):
    print(f"  {bt}: {count:,}")

Input sequence provided is already in string format. No operation performed


Input sequence provided is already in string format. No operation performed


Querying mygene for 28,320 Ensembl IDs...


14 input query terms found dup hits:	[('ENSG00000280018', 2), ('ENSG00000234162', 2), ('ENSG00000175711', 2), ('ENSG00000275496', 2), ('E


1006 input query terms found no hit:	['ENSG00000279374', 'ENSG00000269028', 'ENSG00000267279', 'ENSG00000238758', 'ENSG00000281684', 'ENS


Mapped: 20,240 / 28,320
Not found in mygene: 1,006
Duplicate hits: 14

Biotype distribution:
  protein-coding: 15,465
  ncRNA: 2,448
  pseudo: 2,135
  snoRNA: 161
  snRNA: 14
  other: 7
  tRNA: 6
  rRNA: 2
  scRNA: 2


In [6]:
# Map biotype to Hubstenberger data
hub["gene_biotype"] = hub["ensembl_id"].map(biotype_map)

mapped_count = hub["gene_biotype"].notna().sum()
unmapped_count = hub["gene_biotype"].isna().sum()
print(f"Mapped to mygene biotype: {mapped_count:,}")
print(f"Unmapped (not in mygene): {unmapped_count:,}")

if unmapped_count > 0:
    print(f"\nUnmapped Ensembl IDs (first 10):")
    unmapped_rows = hub[hub["gene_biotype"].isna()]
    for _, row in unmapped_rows.head(10).iterrows():
        print(f"  {row['ensembl_id']} ({row.get('Associated Gene Name', '?')})")

Mapped to mygene biotype: 20,240
Unmapped (not in mygene): 8,081

Unmapped Ensembl IDs (first 10):
  ENSG00000226532 (RP5-1052M9.1)
  ENSG00000248870 (CTD-2015A6.2)
  ENSG00000262074 (SNORD3B-2)
  ENSG00000260475 (RP11-85A1.3)
  ENSG00000279374 (AL596220.1)
  ENSG00000223967 (RP11-505P4.5)
  ENSG00000273295 (AP000350.5)
  ENSG00000202198 (RN7SK)
  ENSG00000268093 (AC022154.7)
  ENSG00000279551 (RP11-588G21.1)


## 4. Filter to Protein-Coding Genes

In [7]:
# Filter to protein-coding (mygene uses "protein-coding" with a hyphen)
hub_pc = hub[hub["gene_biotype"] == "protein-coding"].copy()

print(f"Original Hubstenberger rows: {len(hub):,}")
print(f"Protein-coding genes: {len(hub_pc):,}")
print(f"Genes lost: {len(hub) - len(hub_pc):,}")

# Biotype distribution of excluded genes
lost = hub[(hub["gene_biotype"] != "protein-coding") & hub["gene_biotype"].notna()]
print(f"\nBiotypes of excluded genes (mapped but not protein-coding):")
for bt, count in lost["gene_biotype"].value_counts().head(10).items():
    print(f"  {bt}: {count:,}")

unmapped_lost = hub[hub["gene_biotype"].isna()]
print(f"\nUnmapped (no mygene hit, excluded): {len(unmapped_lost):,}")

Original Hubstenberger rows: 28,321
Protein-coding genes: 15,465
Genes lost: 12,856

Biotypes of excluded genes (mapped but not protein-coding):
  ncRNA: 2,448
  pseudo: 2,135
  snoRNA: 161
  snRNA: 14
  other: 7
  tRNA: 6
  rRNA: 2
  scRNA: 2

Unmapped (no mygene hit, excluded): 8,081


## 5. Validation Against Jason's Reference

Our protein-coding set should be a superset of the 12,544 genes in Jason's final
merged CSV. Using `mygene` (same NCBI source as `org.Hs.eg.db`) should reduce the
missing gene count from 38 (GENCODE v19) to ~6.

In [8]:
# Load reference to get the 12,544 Ensembl IDs
ref = pd.read_csv(REFERENCE_DIR / "12544_Hub_CAGE_MERGE_with_CapIndex.csv")
ref_ids = set(ref["ensembl_id"].unique())
our_ids = set(hub_pc["ensembl_id"].unique())

print(f"Our protein-coding gene count: {len(our_ids):,}")
print(f"Jason's reference gene count: {len(ref_ids):,}")

# Superset check
in_ref_not_ours = ref_ids - our_ids
in_ours_not_ref = our_ids - ref_ids
overlap = ref_ids & our_ids

print(f"\nOverlap: {len(overlap):,}")
print(f"In reference but NOT in our protein-coding set: {len(in_ref_not_ours)}")
print(f"In our set but NOT in reference (expected ~2,148 Hub-only genes): {len(in_ours_not_ref):,}")

if in_ref_not_ours:
    print(f"\nWARNING: {len(in_ref_not_ours)} reference genes missing from our set:")
    for eid in sorted(in_ref_not_ours):
        # Look up gene name in reference
        gene_name = ref[ref["ensembl_id"] == eid]["Associated Gene Name"].values
        name = gene_name[0] if len(gene_name) > 0 else "unknown"
        print(f"  {eid} ({name})")
else:
    print("\nSUCCESS: All 12,544 reference genes are in our protein-coding set.")

Our protein-coding gene count: 15,465
Jason's reference gene count: 12,544

Overlap: 12,538
In reference but NOT in our protein-coding set: 6
In our set but NOT in reference (expected ~2,148 Hub-only genes): 2,927

  ENSG00000112096 (SOD2)
  ENSG00000153113 (CAST)
  ENSG00000168078 (PBK)
  ENSG00000179818 (PCBP1-AS1)
  ENSG00000189144 (ZNF573)
  ENSG00000215271 (HOMEZ)


## 6. Save Output

In [9]:
# Save protein-coding genes
output_path = PROCESSED_DIR / "02_protein_coding_genes.csv"
hub_pc.to_csv(output_path, index=False)

print(f"Saved {len(hub_pc):,} protein-coding genes to {output_path}")
print(f"Columns: {list(hub_pc.columns)}")

Saved 15,465 protein-coding genes to ../data/processed/02_protein_coding_genes.csv
Columns: ['Ensembl Gene ID', 'Associated Gene Name', 'RNA enrichment in P-body (log2) (Fold change=sorted P-bodies/pre-sorted fraction)', 'p-value of RNA enrichment ', 'adjusted p-value (FDR) of RNA enrichment ', 'sorted P-body replicate 1 (count per million reads (CPM))', 'sorted P-body replicate 2 (count per million reads (CPM))', 'sorted P-body replicate 3 (count per million reads (CPM))', 'sorted P-body replicate average (count per million reads (CPM))', 'pre-sorted fraction replicate 1 (count per million reads (CPM))', 'pre-sorted fraction replicate 2 (count per million reads (CPM))', 'pre-sorted fraction replicate 3 (count per million reads (CPM))', 'pre-sorted fraction replicate average (count per million reads (CPM))', 'sorted P-body replicate 1 (mapped read number)', 'sorted P-body replicate 2 (mapped read number)', 'sorted P-body replicate 3 (mapped read number)', 'pre-sorted fraction replicate